# Exploratory Data Analysis (EDA)

In [1]:
import pandas as pd

### Loading and preprocessing of the data using the `loading.py` module

In [2]:
%load_ext autoreload
%autoreload 2

from asrs_lora_classifier.loader import load_all

df = load_all()
df.shape

(85042, 7)

### Freeze the label set

In [3]:
train = df[df["year"] <= 2022]
counts = train["labels"].explode().value_counts()
len(counts), (counts >= 200).sum()

(109, np.int64(54))

In [4]:
counts[(counts >= 100) & (counts < 400)]

labels
Ground Event / Encounter Ground Equipment Issue                              382
Ground Event / Encounter Vehicle                                             376
Inflight Event / Encounter Object                                            305
Inflight Event / Encounter Bird / Animal                                     299
Ground Excursion Taxiway                                                     296
Inflight Event / Encounter VFR In IMC                                        267
Deviation / Discrepancy - Procedural Security                                263
Deviation / Discrepancy - Procedural Landing Without Clearance               250
Ground Event / Encounter Aircraft                                            237
Ground Event / Encounter Gear Up Landing                                     222
Ground Event / Encounter Weather / Turbulence                                220
No Specific Anomaly Occurred Unwanted Situation                              213
Deviation / Discrepan

### Check that a label count of 200 is a good cutoff for the labels we will keep

In [5]:
keep = counts[counts >= 200].index
val = df[df["year"] == 2023]
val_counts = val["labels"].explode().value_counts()
val_counts.reindex(keep, fill_value=0).sort_values().head(10)

labels
Deviation / Discrepancy - Procedural Landing Without Clearance     2
No Specific Anomaly Occurred All Types                             5
Deviation / Discrepancy - Procedural Security                      7
Inflight Event / Encounter VFR In IMC                              9
Deviation / Discrepancy - Procedural Other / Unknown              10
Ground Event / Encounter Aircraft                                 13
Flight Deck / Cabin / Aircraft Event Passenger Misconduct         17
Ground Event / Encounter Gear Up Landing                          18
Inflight Event / Encounter Object                                 23
Ground Event / Encounter Vehicle                                  25
Name: count, dtype: int64

We apply a new rule with two conditions:
1. ≥ 200 in train (enough to learn)
2. ≥ 20 in validation (enough to measure)

In [6]:
val_aligned = val_counts.reindex(counts.index, fill_value=0)
rule = (counts >= 200) & (val_aligned >= 20)
final_labels = counts[rule].index.tolist()
len(final_labels)

46

In [7]:
exploded = df.explode("labels")
per_year = pd.crosstab(exploded["labels"], exploded["year"])
per_year.loc[["Deviation / Discrepancy - Procedural Landing Without Clearance",
              "No Specific Anomaly Occurred All Types",
              "No Specific Anomaly Occurred Unwanted Situation"]]

year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
labels,,,,,,,,,,,,,,,,
Deviation / Discrepancy - Procedural Landing Without Clearance,17,19,10,13,17,33,23,16,28,25,29,10,10,2,7,6
No Specific Anomaly Occurred All Types,81,45,44,51,80,80,25,20,22,31,44,4,2,5,2,5
No Specific Anomaly Occurred Unwanted Situation,0,0,0,0,0,0,0,0,0,0,39,93,81,87,62,92
